# 📘 Fincept Notebook — Dollar-Cost Averaging Simulator

**Investing · Beginner · ~12 min · Standard library only**

Dollar-cost averaging (DCA) means investing a fixed dollar amount on a regular schedule — say $500 every month — regardless of price. You buy more units when prices are low and fewer when prices are high. This notebook simulates DCA over a 24-month price series for an index fund and compares it against putting the same total in as a single lump sum on day one.

**What you'll learn**
- How to track units, cost basis, and portfolio value month by month under DCA
- How DCA's average cost per unit differs from the simple average price
- When DCA wins (volatile or declining entry) versus when lump-sum wins (steadily rising market)


## 1. The price series

Here is our embedded 24-month month-end price series for a hypothetical index fund. It dips early, bottoms around month 8, then recovers — a realistic 'choppy then rising' path. Every code cell below re-embeds this same list so it can run on its own.


In [ ]:
# Month-end prices for the index fund (month 0 .. month 23)
prices = [
    100.00,  98.50,  95.20,  92.80,  90.10,  93.40,  88.70,  85.30,
     82.90,  86.50,  89.20,  91.80,  94.30,  92.10,  96.70,  99.40,
    101.20, 103.80, 102.50, 105.90, 108.30, 110.10, 112.70, 115.00,
]

print(f"Months in series : {len(prices)}")
print(f"Start price (m0) : ${prices[0]:>7.2f}")
print(f"Lowest price     : ${min(prices):>7.2f}  (month {prices.index(min(prices))})")
print(f"Highest price    : ${max(prices):>7.2f}  (month {prices.index(max(prices))})")
print(f"End price (m23)  : ${prices[-1]:>7.2f}")

simple_avg = sum(prices) / len(prices)
print(f"\nSimple average price over the 24 months: ${simple_avg:.2f}")
total_change = (prices[-1] / prices[0] - 1) * 100
print(f"Total price change month 0 -> month 23 : {total_change:+.1f}%")


## 2. Run the DCA simulation

We invest a fixed **$500 every month**. Each month, units bought = contribution / price. We accumulate units, total invested, and portfolio value (units held × current price). The **average cost per unit** is total invested ÷ units held — the number that really matters.


In [ ]:
prices = [
    100.00,  98.50,  95.20,  92.80,  90.10,  93.40,  88.70,  85.30,
     82.90,  86.50,  89.20,  91.80,  94.30,  92.10,  96.70,  99.40,
    101.20, 103.80, 102.50, 105.90, 108.30, 110.10, 112.70, 115.00,
]

MONTHLY = 500.00  # fixed dollar contribution each month

units = 0.0
invested = 0.0

header = f"{'Mo':>3} {'Price':>8} {'Buy$':>7} {'Units+':>8} {'Units':>9} {'Invested':>10} {'Value':>10} {'AvgCost':>8}"
print(header)
print("-" * len(header))

for m, p in enumerate(prices):
    bought = MONTHLY / p
    units += bought
    invested += MONTHLY
    value = units * p
    avg_cost = invested / units
    # print every 3rd month plus the last, to keep the table compact
    if m % 3 == 0 or m == len(prices) - 1:
        print(f"{m:>3} {p:>8.2f} {MONTHLY:>7.0f} {bought:>8.3f} "
              f"{units:>9.3f} {invested:>10.2f} {value:>10.2f} {avg_cost:>8.2f}")

print("-" * len(header))
print(f"\nDCA final units held    : {units:.3f}")
print(f"DCA total invested      : ${invested:,.2f}")
print(f"DCA average cost / unit : ${invested / units:.2f}")
print(f"DCA final value (@m23)  : ${units * prices[-1]:,.2f}")


## 3. DCA vs lump-sum

Now compare. The lump-sum investor puts the **entire 24 months of contributions in at month 0** — 24 × $500 = $12,000 — and holds. Both end with the same dollars contributed; the only difference is *when*. We compare final value, profit, and return.


In [ ]:
prices = [
    100.00,  98.50,  95.20,  92.80,  90.10,  93.40,  88.70,  85.30,
     82.90,  86.50,  89.20,  91.80,  94.30,  92.10,  96.70,  99.40,
    101.20, 103.80, 102.50, 105.90, 108.30, 110.10, 112.70, 115.00,
]

MONTHLY = 500.00
n = len(prices)
total_capital = MONTHLY * n  # same total dollars for both strategies
end_price = prices[-1]

# --- DCA ---
dca_units = sum(MONTHLY / p for p in prices)
dca_value = dca_units * end_price
dca_avg_cost = total_capital / dca_units

# --- Lump sum: all capital at month 0 price ---
ls_units = total_capital / prices[0]
ls_value = ls_units * end_price
ls_avg_cost = prices[0]

def pct(value, cost):
    return (value / cost - 1) * 100

rows = [
    ("Total invested",   f"${total_capital:,.2f}",            f"${total_capital:,.2f}"),
    ("Units acquired",   f"{dca_units:,.3f}",                 f"{ls_units:,.3f}"),
    ("Avg cost / unit",  f"${dca_avg_cost:,.2f}",             f"${ls_avg_cost:,.2f}"),
    ("Final value",      f"${dca_value:,.2f}",                f"${ls_value:,.2f}"),
    ("Profit",           f"${dca_value - total_capital:,.2f}", f"${ls_value - total_capital:,.2f}"),
    ("Return",           f"{pct(dca_value, total_capital):+.1f}%", f"{pct(ls_value, total_capital):+.1f}%"),
]

print(f"{'Metric':<16} {'DCA':>14} {'Lump sum':>14}")
print("-" * 46)
for label, a, b in rows:
    print(f"{label:<16} {a:>14} {b:>14}")

print("-" * 46)
diff = ls_value - dca_value
winner = "Lump sum" if diff > 0 else "DCA"
print(f"\nWinner in THIS series: {winner} by ${abs(diff):,.2f}")


## 4. When does each strategy win?

The result is path-dependent. To see why, we test the same DCA-vs-lump-sum logic on three stylised 12-month paths: a steadily **rising** market, a steadily **declining** market, and a **V-shaped** dip-and-recover. Watch which strategy wins in each.


In [ ]:
def simulate(prices, monthly=500.0):
    n = len(prices)
    capital = monthly * n
    end = prices[-1]
    dca_units = sum(monthly / p for p in prices)
    ls_units = capital / prices[0]
    return capital, dca_units * end, ls_units * end

# Three 12-month regimes (start at 100 in each)
rising    = [100 + 2.0 * i for i in range(12)]                       # straight up
declining = [100 - 2.0 * i for i in range(12)]                       # straight down
v_shaped  = [100, 95, 90, 85, 80, 78, 80, 85, 90, 95, 100, 105]      # dip then recover

scenarios = [("Rising", rising), ("Declining", declining), ("V-shaped", v_shaped)]

print(f"{'Regime':<11} {'DCA value':>12} {'Lump value':>12} {'Winner':>10}")
print("-" * 47)
for name, series in scenarios:
    cap, dca_v, ls_v = simulate(series)
    winner = "Lump sum" if ls_v > dca_v else "DCA"
    print(f"{name:<11} {dca_v:>12,.2f} {ls_v:>12,.2f} {winner:>10}")

print("\nTakeaways:")
print("  - Rising market : lump sum wins. Money invested earlier rides the whole climb.")
print("  - Declining      : DCA wins. Later buys are cheaper, lowering average cost.")
print("  - V-shaped       : DCA wins. It loads up near the bottom; lump-sum bought at the top.")
print("\nRule of thumb: lump-sum has the higher EXPECTED return because markets rise")
print("more often than they fall, but DCA reduces timing risk and is easier to stick to.")


---
*— Fincept Notebook · part of Fincept Terminal. Edit any cell and press Ctrl+Enter to run.*
